# CNN на MNIST: от свёртки до residual-связи

На изображениях цифр проверим, как `padding` и `stride` меняют карту признаков, обучим несколько небольших CNN и сравним их на одной выборке. У вас будет несколько коротких фрагментов кода и эксперименты с готовыми моделями. Данные MNIST загружаются через `torchvision`.

При выборе архитектуры смотрим на validation; test используем один раз в конце. Дополнительные опыты собраны после основного сравнения.


## 0. Данные и правила опыта

Из официальной train-части берём по 300 цифр каждого класса для обучения и по 100 других для validation. Из официальной test-части — по 100 на класс для финальной проверки. Все модели получат одно разбиение, порядок мини-батчей, оптимизатор и число эпох. Это сравнение архитектур; размеры моделей могут отличаться, поэтому вывод о «лучшей архитектуре вообще» делать нельзя.

In [ ]:
import os, random, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = Path(os.environ.get('MNIST_DATA_DIR', 'mnist_data'))
raw_train = MNIST(DATA_ROOT, train=True, download=True)
raw_test = MNIST(DATA_ROOT, train=False, download=True)

def balanced_indices(labels, n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.cat([torch.where(labels == k)[0][torch.randperm(int((labels == k).sum()), generator=g)[:n]]
                      for k in range(10)])

selected = balanced_indices(raw_train.targets, 400, SEED)
train_idx = torch.cat([selected[400*k:400*k+300] for k in range(10)])
val_idx = torch.cat([selected[400*k+300:400*(k+1)] for k in range(10)])
test_idx = balanced_indices(raw_test.targets, 100, SEED)

def take(raw, idx):
    return raw.data[idx].float().div(255).unsqueeze(1), raw.targets[idx].long()

x_train, y_train = take(raw_train, train_idx)
x_val, y_val = take(raw_train, val_idx)
x_test, y_test = take(raw_test, test_idx)
assert not set(train_idx.tolist()) & set(val_idx.tolist())

def make_loader(x, y, shuffle=False, seed=SEED, batch_size=128):
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)

print('Устройство:', DEVICE, '| train/val/test:', len(y_train), len(y_val), len(y_test))
print('Форма одного изображения:', tuple(x_train[0].shape))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for k, ax in enumerate(axes.flat):
    ax.imshow(x_train[k*300, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Класс {k}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 1. Геометрия свёртки

Возьмём игрушечное изображение 5×5 с активной точкой у границы и фильтр 3×3 из единиц. При `padding=0` карта становится меньше: фильтр нельзя поставить центром на крайний пиксель. При `padding=1` вокруг картинки добавляется рамка из нулей. `stride=2` означает, что фильтр прыгает через одну позицию.

In [ ]:
toy = torch.zeros(1, 1, 5, 5)
toy[0, 0, 0, 0] = 1
toy[0, 0, 2, 2] = 1
kernel = torch.ones(1, 1, 3, 3)

def show_maps(maps):
    fig, axes = plt.subplots(1, len(maps), figsize=(3.3*len(maps), 3))
    for ax, (name, value) in zip(np.atleast_1d(axes), maps.items()):
        ax.imshow(value[0, 0], cmap='Blues', vmin=0, vmax=3)
        ax.set_title(f'{name}: {tuple(value.shape[-2:])}')
        ax.set_xticks(range(value.shape[-1])); ax.set_yticks(range(value.shape[-2]))
        ax.grid(alpha=.25)
    plt.tight_layout(); plt.show()

show_maps({'вход': toy})

**Задание 1. Без padding.** Получите выход свёртки 3×3 на `toy` без рамки и проверьте форму.

In [ ]:
valid = None
assert valid.shape == (1, 1, 3, 3)
show_maps({'вход': toy, 'padding=0': valid})

**С padding.** Повторите с `padding=1`. Какая точка у границы видна теперь?

In [ ]:
same = F.conv2d(toy, kernel, padding=1)
assert same.shape == toy.shape
show_maps({'padding=0': valid, 'padding=1': same})

**Задание 2. Шаг свёртки.** Сохраните `padding=1`, поставьте `stride=2`. До запуска предскажите размер.

In [ ]:
stride2 = None
assert stride2.shape == (1, 1, 3, 3)
show_maps({'stride=1': same, 'stride=2': stride2})

**Задание 3. Формула размера.** Напишите формулу для квадратной карты при dilation=1: `floor((n+2p-k)/s)+1`.

In [ ]:
def output_size(n, k, p, s):
    return None

assert output_size(5, 3, 0, 1) == 3
assert output_size(5, 3, 1, 1) == 5
assert output_size(5, 3, 1, 2) == 3
assert output_size(28, 3, 1, 2) == 14

**Обсудим.** `padding=1` сохраняет размер при ядре 3 и шаге 1, но не гарантирует сохранение информации у границы: в рамке стоят искусственные нули. Сравните `MaxPool2d(2)` и свёртку `stride=2`: оба уменьшают карту, но во втором случае веса обучаются.

Посмотрим на настоящей цифре фильтр, который реагирует на горизонтальные переходы яркости. Знак ответа фильтра важен; для картинки покажем модуль.

In [ ]:
edge_kernel = torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]]).reshape(1, 1, 3, 3)
digit = x_train[10:11]
edge = F.conv2d(digit, edge_kernel, padding=1)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(digit[0, 0], cmap='gray'); axes[0].set_title('Исходная цифра')
axes[1].imshow(edge[0, 0].abs(), cmap='magma'); axes[1].set_title('|Ответ фильтра|')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Цикл обучения

Модель возвращает **логиты** `(B, 10)`. `CrossEntropyLoss` сам вычисляет нужные вероятности в устойчивой численной форме, поэтому `softmax` перед ним не нужен. Сначала реализуем три короткие функции, затем проверим, что модель способна запомнить один батч.

In [ ]:
class PlainCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16*7*7, 10))
    def forward(self, x):
        return self.head(self.features(x))

probe = PlainCNN()
print('Формы:', tuple(x_train[:4].shape), '→', tuple(probe.features(x_train[:4]).shape),
      '→', tuple(probe(x_train[:4]).shape))

### Как меняется изображение внутри CNN

На первом слое видны локальные линии и края. После pooling карта меньше: в ней сохраняются отклики фильтров. Сравните форму входа, первой и второй карты.


In [ ]:
example = x_train[17:18]
visual_model = PlainCNN().eval()
with torch.no_grad():
    first = visual_model.features[:2](example)
    second = visual_model.features(example)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes[0, 0].imshow(example[0, 0], cmap='gray'); axes[0, 0].set_title('вход 28×28')
for j in range(4):
    axes[0, j+1].imshow(first[0, j], cmap='magma')
    axes[0, j+1].set_title(f'канал {j+1}: {first.shape[-1]}×{first.shape[-1]}')
for j in range(5):
    axes[1, j].imshow(second[0, j], cmap='magma')
    axes[1, j].set_title(f'канал {j+1}: {second.shape[-1]}×{second.shape[-1]}')
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()


**Задание 4. Потеря от логитов.** Верните логиты и cross-entropy по одному батчу.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
def logits_and_loss(model, xb, yb):
    logits = model(xb)
    loss = None
    return logits, loss

xb, yb = next(iter(make_loader(x_train, y_train)))
logits, loss = logits_and_loss(probe, xb, yb)
assert logits.shape == (len(xb), 10) and loss.ndim == 0
print('Начальная потеря:', round(loss.item(), 3))

**Задание 5. Шаг оптимизатора.** Три действия: обнулить градиенты, `backward`, `step`.

In [ ]:
def backward_step(optimizer, loss):
    pass

tiny_model = PlainCNN().to(DEVICE)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
before = tiny_model.features[0].weight.detach().clone()
tiny_logits, tiny_loss = logits_and_loss(tiny_model, xb.to(DEVICE), yb.to(DEVICE))
backward_step(tiny_opt, tiny_loss)
assert not torch.equal(before, tiny_model.features[0].weight.detach()), 'Веса должны измениться'

**Число верных ответов.** Укажите ось классов и сравните argmax с метками.

In [ ]:
def count_correct(logits, yb):
    return int((logits.argmax(dim=1) == yb).sum().item())

assert 0 <= count_correct(tiny_logits, yb.to(DEVICE)) <= len(yb)

In [ ]:
def run_epoch(model, data_loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum = correct = count = 0
    for xb, yb in data_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.set_grad_enabled(training):
            logits, loss = logits_and_loss(model, xb, yb)
            if training: backward_step(optimizer, loss)
        loss_sum += loss.item() * len(yb)
        correct += count_correct(logits, yb)
        count += len(yb)
    return loss_sum / count, correct / count

# Диагностика: 12 проходов по одному и тому же батчу.
torch.manual_seed(SEED)
overfit = PlainCNN().to(DEVICE)
opt = torch.optim.Adam(overfit.parameters(), lr=0.01)
batch_idx = torch.cat([torch.arange(k*300, k*300+12) for k in range(10)])
one_batch = make_loader(x_train[batch_idx], y_train[batch_idx], batch_size=len(batch_idx))
assert set(y_train[batch_idx].tolist()) == set(range(10))
one_batch_losses = [run_epoch(overfit, one_batch, opt)[0] for _ in range(12)]
print('Loss одного батча:', round(one_batch_losses[0], 3), '→', round(one_batch_losses[-1], 3))
assert one_batch_losses[-1] < one_batch_losses[0]

**Вопрос.** Если сеть запомнила один сбалансированный батч, означает ли это, что она хорошо работает на других почерках? Что мы проверили этим тестом?

## 3. Архитектуры: pooling, stride и residual

`PlainCNN` использует `padding=1` и max pooling. `NoPaddingCNN` убирает нулевую рамку, так что карта перед классификатором меньше. `StridedCNN` уменьшает карты обучаемыми свёртками с `stride=2`. В residual-блоке путь `x` добавляется к преобразованному пути `F(x)`: выход `ReLU(x + F(x))`. Для сложения формы должны совпадать.

In [ ]:
class StridedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16*7*7, 10))
    def forward(self, x):
        return self.head(self.features(x))

print('Plain:', tuple(PlainCNN().features(x_train[:2]).shape),
      '| Strided:', tuple(StridedCNN().features(x_train[:2]).shape))

In [ ]:
class NoPaddingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=0), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=0), nn.ReLU(), nn.MaxPool2d(2))
        # 28 → 26 → 13 → 11 → 5
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16*5*5, 10))
    def forward(self, x):
        return self.head(self.features(x))

print('padding=1:', tuple(PlainCNN().features(x_train[:2]).shape),
      '| padding=0:', tuple(NoPaddingCNN().features(x_train[:2]).shape))

**Сложение двух путей.** Проверьте на тензорах, что residual-связь сохраняет форму.

In [ ]:
shortcut = torch.randn(2, 8, 28, 28)
residual = torch.randn(2, 8, 28, 28)
combined = F.relu(shortcut + residual)
assert combined.shape == shortcut.shape
assert combined.min() >= 0

**Задание 6. ResidualBlock.** Заполните одну строку `forward`: две свёртки, ReLU между ними, сложение с `x`, ещё ReLU.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x):
        return None

block = ResidualBlock(8)
assert block(torch.randn(2, 8, 28, 28)).shape == (2, 8, 28, 28)

In [ ]:
class TinyResNet(nn.Module):
    """Учебная residual-CNN, не torchvision ResNet-18."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            ResidualBlock(8),
            nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.ReLU(),
            ResidualBlock(16),
            nn.Conv2d(16, 16, 3, stride=2, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16*7*7, 10))
    def forward(self, x):
        return self.head(self.features(x))

**Задание 7. Формы четырёх моделей.** Создайте словарь конструкторов и проверьте логиты `(4,10)`.

In [ ]:
ARCHITECTURES = None
for name, factory in ARCHITECTURES.items():
    assert factory()(x_train[:4]).shape == (4, 10)
    print(name, tuple(factory().features(x_train[:4]).shape))

## 4. Сравнение четырёх архитектур

Архитектуры различаются не только связями, но и числом параметров; поэтому более высокая accuracy сама по себе не доказывает пользу residual-связи. Сформулируйте прогноз **до запуска**: кто будет быстрее? У кого больше параметров? Кто точнее после трёх эпох? У `NoPaddingCNN` меняется не только размер карты, но и размер линейного слоя. Почему это мешает приписать разницу в accuracy только padding? Аналогично, residual-сеть отличается от других и числом параметров.

In [ ]:
def fit(factory, seed=SEED, epochs=3, train_x=x_train, train_y=y_train):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(seed)
    model = factory().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    started = time.perf_counter()
    for epoch in range(epochs):
        tl, ta = run_epoch(model, make_loader(train_x, train_y, shuffle=True, seed=seed+epoch), optimizer)
        vl, va = run_epoch(model, make_loader(x_val, y_val))
        for key, value in zip(history, [tl, ta, vl, va]): history[key].append(value)
        print(f'  эпоха {epoch+1}: train={ta:.3f}, validation={va:.3f}')
    return {'model': model, 'history': history, 'seconds': time.perf_counter()-started,
            'seed': seed, 'epochs': epochs}

runs = {}
for name, factory in ARCHITECTURES.items():
    print('Архитектура:', name)
    runs[name] = fit(factory)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for name, run in runs.items():
    h = run['history']
    axes[0].plot(range(1, len(h['train_loss'])+1), h['train_loss'], marker='o', label=name)
    axes[1].plot(range(1, len(h['val_acc'])+1), h['val_acc'], marker='o', label=name)
axes[0].set(title='Train loss', xlabel='Эпоха', ylabel='Loss')
axes[1].set(title='Validation accuracy', xlabel='Эпоха', ylabel='Accuracy')
for ax in axes: ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 5. Цена качества

При одинаковых эпохах сравним не только accuracy, но и число обучаемых параметров и время обучения. Время зависит от конкретного устройства: его можно сравнивать **внутри одного запуска**, но нельзя считать универсальным свойством архитектуры.

**Число параметров.** Считайте только параметры с `requires_grad=True`.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

assert count_parameters(PlainCNN()) > 0

**Задание 8. Таблица эксперимента.** Соберите строки `(имя, параметры, секунды, validation accuracy)`.

In [ ]:
summary = None
assert len(summary) == 4
for name, params, seconds, accuracy in summary:
    print(f'{name:9s} | {params:8d} параметров | {seconds:5.1f} с | val {accuracy:.3f}')

**Задание 9. Выбор модели.** Для этого занятия заранее выбран критерий: максимальная accuracy на validation после 3 эпох.

In [ ]:
chosen_name = None
assert chosen_name in runs
print('Выбрана по validation:', chosen_name)

**Ошибки по классам.** Найдите два класса с минимальной долей верных ответов на validation.

In [ ]:
def predict(model, x):
    model.eval()
    predictions = []
    with torch.no_grad():
        for xb, _ in make_loader(x, torch.zeros(len(x), dtype=torch.long)):
            predictions.append(model(xb.to(DEVICE)).argmax(1).cpu())
    return torch.cat(predictions)

val_predictions = predict(runs[chosen_name]['model'], x_val)
class_accuracy = {k: (val_predictions[y_val == k] == y_val[y_val == k]).float().mean().item()
                  for k in range(10)}
hardest = sorted(class_accuracy, key=class_accuracy.get)[:2]
print('Трудные классы:', hardest, '| accuracy:', [round(class_accuracy[k], 3) for k in hardest])

In [ ]:
wrong = torch.where(val_predictions != y_val)[0][:6]
fig, axes = plt.subplots(1, len(wrong), figsize=(2*len(wrong), 2.5))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(x_val[i, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{int(y_val[i])} → {int(val_predictions[i])}'); ax.axis('off')
plt.tight_layout(); plt.show()

**Сравните результаты.** Совпал ли прогноз с результатами? Какой обмен между качеством, числом параметров и временем виден? Какая из разниц может быть связана с архитектурой, а какая — с её размером? Предложите один новый опыт, в котором меняется только один фактор.

### Residual-связь при почти одинаковом числе параметров

У двух сетей ниже одинаковые свёртки и число обучаемых параметров. Отличается только сложение с входом блока. Запустите обе сети с одинаковым seed и сравните validation accuracy. Один запуск не доказывает устойчивое преимущество.


In [ ]:
class PlainBlock(ResidualBlock):
    def forward(self, x):
        return F.relu(self.conv2(F.relu(self.conv1(x))))

def make_without_shortcuts():
    model = TinyResNet()
    for i, layer in enumerate(model.features):
        if isinstance(layer, ResidualBlock):
            model.features[i] = PlainBlock(layer.conv1.in_channels)
    return model

print('Параметров:', count_parameters(TinyResNet()), count_parameters(make_without_shortcuts()))
pair = {'со связью': fit(TinyResNet, seed=123),
        'без связи': fit(make_without_shortcuts, seed=123)}
for name, run in pair.items():
    print(name, 'validation accuracy:', round(run['history']['val_acc'][-1], 3))


## Дополнительные исследования

### Повтор с другим seed

Запустите те же четыре архитектуры с seed 123. Устойчив ли порядок моделей? Это проверка разброса результата, а не повод выбрать seed с лучшим test.

**Задание 10. Повторные запуски.** Для каждой архитектуры вызовите `fit(factory, seed=123)`.

In [ ]:
repeat_runs = {}
assert set(repeat_runs) == set(ARCHITECTURES)
for name in runs:
    a = runs[name]['history']['val_acc'][-1]
    b = repeat_runs[name]['history']['val_acc'][-1]
    print(name, round(a, 3), '→', round(b, 3), '| разница', round(b-a, 3))

### Устойчивость к шуму

На одном и том же зафиксированном шуме сравните три уже обученные модели. Это **validation**, не test. Здесь проверяется устойчивость без переобучения модели.

**Задание 11. Одинаковый шум.** Создайте шум на CPU с заданным seed и ограничьте пиксели диапазоном [0,1].

In [ ]:
def add_noise(x, sigma, seed=2026):
    g = torch.Generator().manual_seed(seed)
    return None

assert torch.equal(add_noise(x_val, 0.3), add_noise(x_val, 0.3))
assert add_noise(x_val, 0.3).shape == x_val.shape

**Задание 12. Кривая устойчивости.** Оцените accuracy для каждого уровня шума и каждой модели.

In [ ]:
noise_levels = [0, 0.15, 0.30, 0.45, 0.60]
robustness = {name: [] for name in runs}
for sigma in noise_levels:
    noisy_x = add_noise(x_val, sigma)
    for name, run in runs.items():
        pass
assert all(len(v) == len(noise_levels) for v in robustness.values())
for name, accuracies in robustness.items():
    plt.plot(noise_levels, accuracies, marker='o', label=name)
plt.xlabel('Стандартное отклонение шума'); plt.ylabel('Validation accuracy')
plt.grid(alpha=.3); plt.legend(); plt.show()

### ResNet-18 на MNIST

До сих пор `TinyResNet` был нашей учебной residual-CNN. Теперь возьмём `torchvision.models.resnet18(weights=None)` **без загрузки весов**. Исходный ResNet-18 рассчитан на 3-канальные большие изображения; заменим первую свёртку на одноканальную, уберём ранний max pooling и изменим число классов. Это около 11 млн параметров, поэтому на CPU обучим только **одну эпоху на 1000 train-примерах** как опыт по стоимости. При GPU можно провести те же три эпохи на полном учебном поднаборе и сравнить accuracy с остальными моделями. Результаты с разными объёмами данных/числом эпох нельзя ставить в один рейтинг качества.

**Задание 13. Адаптация ResNet-18.** Измените три компонента: `conv1`, `maxpool`, `fc`.

In [ ]:
from torchvision.models import resnet18
def make_resnet18_mnist():
    model = resnet18(weights=None)
    return model

real_resnet = make_resnet18_mnist()
assert real_resnet(x_train[:2]).shape == (2, 10)
print('ResNet-18 параметров:', count_parameters(real_resnet))

In [ ]:
if DEVICE.type == 'cuda':
    print('GPU: три эпохи на том же train. Accuracy можно сравнивать, время — с учётом GPU.')
    resnet_run = fit(make_resnet18_mnist, epochs=3)
else:
    print('CPU: одна эпоха на 1000 train-примерах. Это только замер стоимости.')
    small_idx = torch.cat([torch.arange(k*300, k*300+100) for k in range(10)])
    resnet_run = fit(make_resnet18_mnist, epochs=1, train_x=x_train[small_idx], train_y=y_train[small_idx])
print('ResNet-18 секунд:', round(resnet_run['seconds'], 1),
      '| validation:', round(resnet_run['history']['val_acc'][-1], 3))

**Обсудим:** что было гипотезой, какие условия сохраняли одинаковыми, что показали запуски, почему сравнение с большим ResNet-18 на CPU ограничено, какой следующий опыт стоило бы провести? Только после этого выполните раздел «Финальный test» для заранее выбранной модели.

## 6. Финальный test

Выполните после выбора модели по validation. Модель уже выбрана по validation; test нужен для одной финальной оценки, а не для перенастройки.

In [ ]:
test_loss, test_acc = run_epoch(runs[chosen_name]['model'], make_loader(x_test, y_test))
print(f'Модель {chosen_name}: test accuracy={test_acc:.3f}, test loss={test_loss:.3f}')